# Image segmentation and neural network quantization

Here are all the import statements needed for all the exercises

1️⃣ What is the dataset / task?

X: Sentinel-2 satellite image patches

Shape ≈ (N, H, W, 12) → 12 spectral bands (not just RGB)

Y: corresponding binary mask for each patch

1 = cultivated land

0 = not cultivated

So this is image segmentation:
👉 For each pixel we want a label (farmable / not farmable), not just one label per image.

In [1]:
%tensorflow_version 2.x
import tensorflow as tf
%load_ext tensorboard
import numpy as np
import matplotlib.pyplot as plt

! pip install tensorflow_model_optimization
import tensorflow_model_optimization as tfmot

Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.


Loading the Sentinel 2 dataset. Images and labels are padded to be 256x256 in size and normalized by their maximum value. 40 images are used for the train partition (X_train, Y_train) and 10 for testing (X_test, Y_test)

In [30]:
loaded = np.load('/content/sentinel2.npz')
X = loaded['X'].astype(np.float32)
Y = loaded['Y'].astype(np.float32)
X = np.pad(X, ((0,0),(3,3),(3,3),(0,0)))
Y = np.pad(Y, ((0,0),(3,3),(3,3),(0,0)))
X_train = X[:40]/np.max(X[:40])
X_test = X[40:]/np.max(X[:40])
Y_train_exp = Y[:40]
Y_test_exp = Y[40:]




**[TODO]** Implement the U-net neural network for segmentation as drawn in the lab document.

🔹 Encoder (downsampling path)

Repeated blocks of Conv → BN → ReLU (stride=1) then Conv → BN → ReLU (stride=2) to downsample.
Save some outputs as skip connections.

🔹 Bottleneck
Another Conv → BN → ReLU that works on the lowest-resolution feature map.
This is the most abstract / global information.

🔹 Decoder (upsampling path)
At each level: Upsample → Conv+BN+ReLU → Concatenate with skip → Conv+BN+ReLU.



In [31]:
#prepare labels (Add channel dim)

print("X_train:", X_train.shape)
print("Y_train:", Y_train_exp.shape)

#U-net building blocks
from tensorflow.keras import layers, models
def conv_bn_relu(x, filters, stride=1):
    x = tf.keras.layers.Conv2D(
        filters,
        kernel_size=3,
        strides=stride,
        padding='same',
        use_bias=False
    )(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)
    return x
input_shape = X_train.shape[1:]   # (H, W, 12)

inputs = tf.keras.Input(shape=input_shape)

# ---------- Encoder ----------
# Level 1
e1 = conv_bn_relu(inputs, filters=32, stride=1)   # C s=1
x  = conv_bn_relu(e1,     filters=32, stride=2)   # C s=2  (downsample H,W by 2)

# Level 2
e2 = conv_bn_relu(x,      filters=64, stride=1)   # C s=1
x  = conv_bn_relu(e2,     filters=64, stride=2)   # C s=2  (downsample again)


#hun this means we saved ex1 and ex2 as strips el be2e is all updated on x

# Bottleneck
x  = conv_bn_relu(x,      filters=128, stride=1)  # C s=1

# ---------- Decoder ----------
# Level 1 up
x  = tf.keras.layers.UpSampling2D(size=(2, 2))(x)       # UP x2
x  = conv_bn_relu(x,filters=64, stride=1)     # C s=1
x  = tf.keras.layers.Concatenate()([x, e2])             # CAT with skip from level 2
x  = conv_bn_relu(x,filters=64, stride=1)     # C s=1

# Level 2 up
x  = tf.keras.layers.UpSampling2D(size=(2, 2))(x)       # UP x2
x  = conv_bn_relu(x,filters=32, stride=1)     # C s=1
x  = tf.keras.layers.Concatenate()([x, e1])             # CAT with skip from level 1
x  = conv_bn_relu(x, filters=32, stride=1)     # C s=1

# ---------- Output ----------
# 1-channel mask with sigmoid for binary segmentation
outputs = tf.keras.layers.Conv2D(
    1,
    kernel_size=1,
    activation='sigmoid'
)(x)
#sigmoid because binary result
model = tf.keras.Model(inputs=inputs, outputs=outputs)

#model.summary()


X_train: (40, 256, 256, 12)
Y_train: (40, 256, 256, 1)


**[TODO]** Compile and train the model (might take some time...)

In [36]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, Y_train_exp,
    validation_data=(X_test, Y_test_exp),
    epochs=100,
    batch_size=2
)


Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 9s 92ms/step - accuracy: 0.9110 - loss: 0.2164 - val_accuracy: 0.8537 - val_loss: 0.4008
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.9191 - loss: 0.2065 - val_accuracy: 0.8714 - val_loss: 0.3240
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 38ms/step - accuracy: 0.8953 - loss: 0.2419 - val_accuracy: 0.9209 - val_loss: 0.2342
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.8995 - loss: 0.2370 - val_accuracy: 0.9103 - val_loss: 0.2708
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.9074 - loss: 0.2234 - val_accuracy: 0.8176 - val_loss: 0.4203
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step - accuracy: 0.8921 - loss: 0.2725 - val_accuracy: 0.4984 - val_loss: 0.8199
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 36ms/step - accuracy: 0.9143 - loss: 0.2098 - val_accuracy: 0.8789 - val_loss: 0.2811
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 1s 31ms/step - accuracy: 0.9254 - loss: 0.1824 - val_accuracy: 0.

**[TODO]** Test the model on the test set and measure the accuracy.

In [37]:
test_loss, test_acc = model.evaluate(X_test, Y_test_exp)
print("Test accuracy:", test_acc)
print("Test loss:", test_loss)


#but the dr wants us to do it manually so :
# Predict probability mask
preds = model.predict(X_test)

# Convert to binary mask (0 or 1)
preds_bin = (preds > 0.5).astype(np.float32)

# Pixel-wise accuracy
manual_acc = np.mean(preds_bin == Y_test_exp)

print("Manual test accuracy:", manual_acc)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.9178 - loss: 0.2022
Test accuracy: 0.9178375005722046
Test loss: 0.20221209526062012


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 891ms/step
Manual test accuracy: 0.9178375244140625


**[TODO]** Convert model to TFLite with 8-bit weight quantization

In [38]:
import tensorflow as tf

#This takes your trained U-Net and prepares a TFLite converter object.
converter = tf.lite.TFLiteConverter.from_keras_model(model)

#This tells TensorFlow: “Shrink the model size as much as possible.”
#The optimization triggers automatic 8-bit weight quantization.
converter.optimizations = [tf.lite.Optimize.OPTIMIZE_FOR_SIZE]  # enables weight quantization

#This forces the TFLite model to:Use float32 activations, Only quantize the weights
converter.target_spec.supported_types = [tf.float32]  # keep activations in float

tflite_model = converter.convert()

# Save model
with open("unet_weight_quant.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite conversion complete!")


Saved artifact at '/tmp/tmpclwrl23t'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 256, 12), dtype=tf.float32, name='keras_tensor_198')
Output Type:
  TensorSpec(shape=(None, 256, 256, 1), dtype=tf.float32, name=None)
Captures:
  137630419456400: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137630806231760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137630806230032: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137630806232912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137630806231568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137630806234832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137630806229840: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137630806229264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137630806231376: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137630806232336: TensorSpec(shape=(), dtype=tf.resource, name=None)
 

TFLite conversion complete!


**[TODO]** Test the accuracy of the quantized model by writing your own "evaluate" function. Remember that TFLite interpreter can only process one sample at a time, not a batch.

In [39]:
import numpy as np
import tensorflow as tf

def evaluate_tflite_model(tflite_path, X_test, Y_test):
    # Load TFLite model
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Squeeze masks if needed
    Y_test_squeezed = np.squeeze(Y_test)  # (N,H,W)

    total_pixels = 0
    correct_pixels = 0

    # Loop through each sample (TFLite cannot batch)
    #Since TFLite cannot process batches, we process ONE image at a time:
    for i in range(len(X_test)):
        x = X_test[i:i+1]  # shape (1,H,W,C)keeps the batch dimension.

        # Set input tensor, This is how you pass data into the model.
        interpreter.set_tensor(input_details[0]['index'], x)

        # Run inference, This is the equivalent of model.predict().It runs the model ONCE on the image.
        interpreter.invoke()

        # Get prediction
        pred = interpreter.get_tensor(output_details[0]['index'])  # (1,H,W,1)The model outputs probabilities between 0 and 1
        pred = np.squeeze(pred)  # (H,W)Squeezing removes useless dimensions

        # Convert probabilities → convert to binary mask
        pred_bin = (pred > 0.5).astype(np.float32)
'''our segmentation is binary → cultivated or not.
Threshold at 0.5 gives:
1 = cultivated
0 = not cultivated'''

        # Ground truth for this sample
        y_true = Y_test_squeezed[i]  # (H,W)

        # Compare prediction & ground truth
        correct_pixels += np.sum(pred_bin == y_true)
        total_pixels   += y_true.size

    # Final pixel-wise accuracy
    return correct_pixels / total_pixels

tflite_accuracy = evaluate_tflite_model(
    "unet_weight_quant.tflite",
    X_test,
    Y_test_exp
)

print("Quantized TFLite model accuracy:", tflite_accuracy)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Quantized TFLite model accuracy: 0.9178375244140625


📌Could accuracy EVER improve?

Technically, yes but RARELY.

If the model was slightly overfitting, quantization (which adds noise) can sometimes act as a regularizer → slightly improving generalization.

But this is uncommon.

In 95% of cases:

Quantization keeps accuracy the same or reduces it a little.

**[TODO]** Finetune the Keras model using quantization-aware training and measure the accuracy on the test set after actually quantizing it